# 인천·경기·충남 폐기업체 데이터 통합

전국폐기물처리업소표준데이터(notebook 01)에서는 소각장 필터링이 어려웠으므로,
서울 인근 3개 지역(경기도, 충청남도, 인천)의 개별 데이터를 수집하여 통합합니다.

통합 컬럼: `지역`, `시설명`, `주소`, `시설용량`, `처리대상폐기물`, `비고`

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('../../data/raw/소각장/폐기업체_데이터')
OUTPUT_DIR = Path('../../outputs/소각장')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 파일 확인
for f in sorted(DATA_DIR.iterdir()):
    print(f'  {f.name}')

  경기도_폐기물처리업체현황.csv
  인천_2024_폐기물 처리업체 현황_(생활, 사업장일반, 건설).xlsx
  전국폐기물처리업소표준데이터.csv
  충청남도_폐기물 처리시설 현황(소각시설).csv


## 1. 경기도

경기도 데이터에서 소각 업체(처리업종구분명 `②ⓓ`)만 필터링.

In [2]:
gg = pd.read_csv(DATA_DIR / '경기도_폐기물처리업체현황.csv', encoding='cp949')
print(f'원본: {gg.shape}')
print(f'컬럼: {gg.columns.tolist()}')
print(f'\n처리업종구분명 분포:')
print(gg['처리업종구분명'].value_counts())

원본: (6661, 10)
컬럼: ['시군명', '사업장명', '전화번호', '처리업종구분명', '처리대상폐기물정보', '소재지우편번호', '소재지지번주소', '소재지도로명주소', 'WGS84위도', 'WGS84경도']

처리업종구분명 분포:
처리업종구분명
⑧     1611
①ⓑ    1576
⑦     1312
①ⓐ    1107
①ⓒ     547
⑤      336
⑨      101
⑥       40
②ⓓ      16
②ⓔ       9
②ⓕ       5
③        1
Name: count, dtype: int64


In [3]:
# 소각 업체만 필터 (②ⓓ = 폐기물중간처분업)
gg_소각 = gg[gg['처리업종구분명'].fillna('').str.contains('②ⓓ')].copy()
print(f'소각 업체: {len(gg_소각)}개')
display(gg_소각[['사업장명', '소재지도로명주소']].head())

gg_result = pd.DataFrame({
    '지역': '경기도',
    '시설명': gg_소각['사업장명'].values,
    '주소': gg_소각['소재지도로명주소'].values,
    '시설용량': None,
    '처리대상폐기물': gg_소각['처리대상폐기물정보'].values,
    '비고': None,
})
print(f'\n경기도 결과: {len(gg_result)}행')

소각 업체: 16개


,사업장명,소재지도로명주소
464,리뉴에너지경기㈜,경기도 광주시 곤지암읍 열미길 22
1270,㈜청송산업개발,경기도 동두천시 삼육사로652번길 60(상패동)
1792,코어엔텍 주식회사,경기도 시흥시 소망공원로 5
1793,시흥도시공사,경기도 시흥시 옥구천서로 9번길 45
2108,㈜비노텍,경기도 안산시 단원구 해안로 308(원시동)



경기도 결과: 16행


## 2. 충청남도

충남 데이터는 이미 소각시설만 포함되어 있음. 합계 행을 제거하고 주소에 시/군명을 보충

In [4]:
cn = pd.read_csv(str(DATA_DIR / '충청남도_폐기물 처리시설 현황(소각시설).csv'), encoding='cp949')
print(f'원본: {cn.shape}')
print(f'컬럼: {cn.columns.tolist()}')
display(cn)

원본: (13, 8)
컬럼: ['구분', '시설명', '위치', '규모(㎏/h)', '시설용량(톤/일)', '설치신고일', '사용개시일', '비고']


,구분,시설명,위치,규모(㎏/h),시설용량(톤/일),설치신고일,사용개시일,비고
0,합계,11개소,NaN,"45,364.25","1,111.20",NaN,NaN,NaN
1,천안,환경에너지사업소,서북구 백석동 531,"13,334",320,99.08.10,01.11.10,NaN
2,천안,환경에너지사업소,서북구 백석동 531,"10,417",250,13.03.15,15.09.15,NaN
3,공주,공주시청,검상동 326,"2,083",50,00.12.30,01.05.25,NaN
4,보령,생활폐기물종합처리장,남곡동 1140-6,"2,083",50,05.07.01,06.08.31,NaN
5,아산,환경과학공원,배미로 154,"8,333",200,08.10.15,11.06.15,통합허가
6,논산,환경자원화센터,은진면 버들길 137,"2,083",50,04.06.15,06.04.29,NaN
7,계룡,계룡시청,두마면 입암리 516,"1,300",31.2,05.03.23,06.08.05,NaN
8,금산,생활페기물소각시설,추부면 용지리 432-2,1.25,30,17.04.17,19.05.14,NaN
9,서천,자원순환센터,비인면 관리 547-1,"1,250",30,13.12.13,15.12.28,NaN


In [5]:
# 합계 행 제거
cn = cn[cn['구분'] != '합계'].copy()

# 주소 조합: '충청남도 {구분}시/군 {위치}'
def make_cn_address(row):
    gu = str(row['구분']).strip()
    loc = str(row['위치']).strip() if pd.notna(row['위치']) else ''
    suffix = '시' if gu in ['천안', '공주', '보령', '아산', '논산', '계룡'] else '군'
    return f'충청남도 {gu}{suffix} {loc}'.strip()

cn_result = pd.DataFrame({
    '지역': '충청남도',
    '시설명': cn['시설명'].values,
    '주소': cn.apply(make_cn_address, axis=1).values,
    '시설용량': cn['시설용량(톤/일)'].values,
    '처리대상폐기물': None,
    '비고': cn['비고'].values,
})
print(f'충남 결과: {len(cn_result)}행')
display(cn_result)

충남 결과: 12행


,지역,시설명,주소,시설용량,처리대상폐기물,비고
0,충청남도,환경에너지사업소,충청남도 천안시 서북구 백석동 531,320,None,NaN
1,충청남도,환경에너지사업소,충청남도 천안시 서북구 백석동 531,250,None,NaN
2,충청남도,공주시청,충청남도 공주시 검상동 326,50,None,NaN
3,충청남도,생활폐기물종합처리장,충청남도 보령시 남곡동 1140-6,50,None,NaN
4,충청남도,환경과학공원,충청남도 아산시 배미로 154,200,None,통합허가
5,충청남도,환경자원화센터,충청남도 논산시 은진면 버들길 137,50,None,NaN
6,충청남도,계룡시청,충청남도 계룡시 두마면 입암리 516,31.2,None,NaN
7,충청남도,생활페기물소각시설,충청남도 금산군 추부면 용지리 432-2,30,None,NaN
8,충청남도,자원순환센터,충청남도 서천군 비인면 관리 547-1,30,None,NaN
9,충청남도,환경사업소,충청남도 청양군 청양읍 벽천리 109-8,15,None,NaN


## 3. 인천

인천 엑셀에서 소각 관련 3개 시트를 파싱합니다.
- `1-가. 공공소각`: 공공 소각시설
- `2-가. 자가중간처분(소각)`: 자가 소각시설
- `4-가. 중간처분(소각)`: 민간 소각시설

In [6]:
xlsx = str(DATA_DIR / '인천_2024_폐기물 처리업체 현황_(생활, 사업장일반, 건설).xlsx')
xl = pd.ExcelFile(xlsx)
print('전체 시트 목록:')
for s in xl.sheet_names:
    print(f'  - {s}')

전체 시트 목록:
  - 1-가. 공공소각
  - 1-나. 공공기타
  - 1-다. 공공매립
  - 2-가. 자가중간처분(소각)
  - 2-나. 자가중간처분(기타)
  - 2-다. 자가재활용처리시설
  - 2-라. 자가매립
  - 3-가. 수집운반(생활, 사업장일반)
  - 3-나. 수집운반(건설)
  - 4-가. 중간처분(소각)
  - 4-나. 중간처분(기타)
  - 4-다. 중간처리(건설)
  - 5. 최종처분
  - 6. 종합처분
  - 7-가. 폐기물처리(재활용) 신고
  - 7-나. 폐기물처리(수집운반) 신고
  - 8-가. 재활용처리(중간)
  - 8-나. 재활용처리(최종)
  - 8-다. 재활용처리(종합)


In [7]:
incheon_frames = []

target_sheets = ['1-가. 공공소각', '2-가. 자가중간처분(소각)', '4-가. 중간처분(소각)']

for sheet_name in target_sheets:
    d = xl.parse(sheet_name, header=None)
    
    # 헤더는 row 3 (줄바꿈 제거)
    header_row = 3
    headers = [str(h).replace('\n', '').strip() if pd.notna(h) else f'col_{i}'
               for i, h in enumerate(d.iloc[header_row].tolist())]
    d.columns = headers
    d = d.iloc[header_row+1:].copy()
    
    # 시설명/업체명 컬럼 찾기
    name_col = next((c for c in d.columns if c in ['시설명', '업체명']), d.columns[3])
    d = d.dropna(subset=[name_col])
    d = d[d[name_col].astype(str).str.strip() != '']
    
    # 주소: 시군구 + 소재지 조합
    addr_col = next((c for c in d.columns if '소재지' in c), None)
    sigungu_col = next((c for c in d.columns if '시군구' in c), None)
    if sigungu_col and addr_col:
        addr = d.apply(lambda r: f'인천광역시 {r[sigungu_col]} {r[addr_col]}'
                       if pd.notna(r[sigungu_col]) and pd.notna(r[addr_col])
                       and '인천' not in str(r[addr_col])
                       else str(r[addr_col]) if pd.notna(r[addr_col]) else None, axis=1)
    elif addr_col:
        addr = d[addr_col]
    else:
        addr = None
    
    # 각 컬럼 매핑
    cap_col = next((c for c in d.columns if '시설용량' in c), None)
    waste_col = next((c for c in d.columns if '폐기물' in c), None)
    note_col = next((c for c in d.columns if '비고' in c), None)
    
    frame = pd.DataFrame({
        '지역': '인천',
        '시설명': d[name_col].values,
        '주소': addr.values if addr is not None else None,
        '시설용량': d[cap_col].values if cap_col else None,
        '처리대상폐기물': d[waste_col].values if waste_col else None,
        '비고': d[note_col].values if note_col else None,
    })
    incheon_frames.append(frame)
    print(f'[{sheet_name}]: {len(frame)}개')

ic_result = pd.concat(incheon_frames, ignore_index=True)
print(f'\n인천 합계: {len(ic_result)}행')

[1-가. 공공소각]: 10개
[2-가. 자가중간처분(소각)]: 8개
[4-가. 중간처분(소각)]: 6개

인천 합계: 24행


## 4. 3개 지역 통합

In [8]:
combined = pd.concat([gg_result, cn_result, ic_result], ignore_index=True)

print(f'통합 결과: {len(combined)}행 x {len(combined.columns)}열')
print(f'컬럼: {combined.columns.tolist()}')
print(f'\n지역별 건수:')
print(combined['지역'].value_counts())
print(f'\n컬럼별 유효/결측:')
for col in combined.columns:
    valid = combined[col].notna().sum()
    null = combined[col].isna().sum()
    print(f'  {col:12s} | 유효: {valid:3d} | 결측: {null:3d}')

통합 결과: 52행 x 6열
컬럼: ['지역', '시설명', '주소', '시설용량', '처리대상폐기물', '비고']

지역별 건수:
지역
인천      24
경기도     16
충청남도    12
Name: count, dtype: int64

컬럼별 유효/결측:
  지역           | 유효:  52 | 결측:   0
  시설명          | 유효:  52 | 결측:   0
  주소           | 유효:  52 | 결측:   0
  시설용량         | 유효:  36 | 결측:  16
  처리대상폐기물      | 유효:  11 | 결측:  41
  비고           | 유효:   1 | 결측:  51


In [9]:
display(combined)

,지역,시설명,주소,시설용량,처리대상폐기물,비고
0,경기도,리뉴에너지경기㈜,경기도 광주시 곤지암읍 열미길 22,None,NaN,None
1,경기도,㈜청송산업개발,경기도 동두천시 삼육사로652번길 60(상패동),None,NaN,None
2,경기도,코어엔텍 주식회사,경기도 시흥시 소망공원로 5,None,NaN,None
3,경기도,시흥도시공사,경기도 시흥시 옥구천서로 9번길 45,None,NaN,None
4,경기도,㈜비노텍,경기도 안산시 단원구 해안로 308(원시동),None,NaN,None
5,경기도,대일개발㈜,경기도 안산시 단원구 지원로 7(성곡동),None,NaN,None
6,경기도,성림유화㈜,경기도 안산시 단원구 첨단로 215(성곡동),None,NaN,None
7,경기도,부경산업㈜,경기도 안산시 단원구 신원로91번길 16(성곡동),None,NaN,None
8,경기도,한국환경개발㈜,경기도 안산시 단원구 첨단로207번길 5(성곡동),None,NaN,None
9,경기도,㈜가나에너지,경기도 양주시 남면 삼일로485번길 92,None,NaN,None


## 5. CSV 저장

In [10]:
output_path = OUTPUT_DIR / 'regional_incinerator_candidates_combined.csv'
combined.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {output_path}')
print(f'  {len(combined)}행 x {len(combined.columns)}열')

저장 완료: ..\..\outputs\eda\regional_incinerator_candidates_combined.csv
  52행 x 6열


## 6. 인사이트 정리

### 데이터 현황
- **경기도 16개**: 시설용량·처리대상폐기물 정보 없음 → 별도 조사 필요
- **충청남도 12개**: 시설용량 보유, 주소가 지번 기반 → 지오코딩 필요
- **인천 24개**: 시설용량 보유, 일부 주소에 '인천광역시' 누락
